# 02 - Baseline (Qwen2.5-1.5B-Instruct, zero-shot)

GPU required. Install packages, clone the repo, regenerate the MedQuAD splits, run zero-shot inference over a fixed test sample, evaluate, save `results/baseline.csv`.

In [ ]:
%pip install -q transformers datasets accelerate evaluate rouge_score pandas pyyaml

In [ ]:
import os
import sys

REPO_URL = "https://github.com/satyazm/Finetuning_LLMs.git"
REPO_DIR = "/kaggle/working/Finetuning_LLMs"

if not os.path.exists(REPO_DIR):
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    !git clone -q {clone_url} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

## Regenerate the dataset

`data/` isn't committed (it's derived), so pull MedQuAD from the HF Hub again here — same `preprocess.py` used locally, same split/seed, so `data/test.json` matches what was verified in `01_preprocessing.ipynb`.

In [ ]:
from src.data.preprocess import run as preprocess_run

preprocess_run(output_dir="data")

## Run baseline

Loads the base model from `configs/model.yaml`, generates on a fixed 200-example sample of the test split (seeded, so LoRA/QLoRA notebooks can reuse the same sample for a fair comparison), computes ROUGE + latency + peak GPU memory, and saves `results/baseline.csv` (+ `results/baseline_summary.json`).

In [ ]:
import yaml

from src.evaluation.evaluate import run_baseline

with open("configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)

df, summary = run_baseline(
    model_name=model_cfg["base_model"],
    test_path="data/test.json",
    output_csv="results/baseline.csv",
)

## Verify

In [ ]:
print(summary)
df.head()